# DATA266 Lab 1 — Task 3 CycleGAN (Google Colab)

Run cells top to bottom on a GPU runtime. The smoke test runs before training. Checkpoints and final artifacts are saved to Google Drive and training resumes automatically after interruption.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_ROOT = Path('/content/GenAI')
if not (REPO_ROOT / 'task3_gan' / 'member_denisha').is_dir():
    if REPO_ROOT.exists(): shutil.rmtree(REPO_ROOT)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/denishatank12/GenAI.git', str(REPO_ROOT)], check=True)
MEMBER_DIR = REPO_ROOT / 'task3_gan' / 'member_denisha'
DATA_ROOT = REPO_ROOT / 'task3_gan' / 'data'
DRIVE_ROOT = Path('/content/drive/MyDrive/DATA266_Lab1_checkpoints/task3')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints_competition'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
(REPO_ROOT / 'reproducibility' / 'raw_logs').mkdir(parents=True, exist_ok=True)
print('Repository:', REPO_ROOT)
print('Persistent checkpoints:', CHECKPOINT_DIR)

In [ ]:
# Install dependencies without replacing Colab's CUDA-compatible torch.
requirements = []
for line in (REPO_ROOT / 'requirements.txt').read_text().splitlines():
    line = line.strip()
    if line and not line.startswith('#') and not line.lower().startswith(('torch=', 'torch>', 'torch<', 'torchvision=', 'torchvision>', 'torchvision<')):
        requirements.append(line)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *requirements], check=True)
import torch
assert torch.cuda.is_available(), 'GPU is not enabled. Choose Runtime > Change runtime type > T4 GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], check=False)

In [ ]:
for name, expected_min in [('monet_jpg', 300), ('photo_jpg', 7000)]:
    folder = DATA_ROOT / name
    images = [p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and not p.name.startswith('._') and p.name != '.DS_Store']
    print(f'{name}: {len(images)} images')
    assert len(images) >= expected_min, f'Missing approved data in {folder}'

## Smoke test

This must pass before the long run starts.

In [ ]:
subprocess.run([sys.executable, 'src/smoke_test.py', '--config', 'config_competition.yaml'], cwd=MEMBER_DIR, check=True)

## Train, checkpoint, resume, and evaluate

The competition configuration is 60 epochs, batch size 4, and 2,000 samples per epoch. Run this cell again after a disconnect; it resumes from the newest Drive checkpoint.

In [ ]:
subprocess.run([
    sys.executable, 'run_task3.py',
    '--config', 'config_competition.yaml',
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--require-gpu'
], cwd=MEMBER_DIR, check=True)

In [ ]:
OUTPUTS = MEMBER_DIR / 'outputs'
SAVED = DRIVE_ROOT / 'final_outputs'
if OUTPUTS.exists(): shutil.copytree(OUTPUTS, SAVED, dirs_exist_ok=True)
log = REPO_ROOT / 'reproducibility' / 'raw_logs' / 'task3_denisha.log'
if log.exists(): shutil.copy2(log, DRIVE_ROOT / log.name)
submission = OUTPUTS / 'submission.csv'
assert submission.exists(), 'submission.csv was not produced'
print(submission.read_text())
print('Saved artifacts to:', DRIVE_ROOT)

## Final manual requirements

Complete the two-rater 30-image audit in `outputs/human_audit.csv`, run `src/audit_agreement.py`, then upload `outputs/submission.csv` to Kaggle. The required format is exactly two rows with columns `ID,FID,MiFID`.